# Full-book PaddleOCR from existing layout regions

This notebook runs PaddleOCR over the complete Pierce book using the committed DocLayout-YOLO regions. It does **not** rerun layout detection and it does not use Chandra text as OCR input.

Enable a Kaggle Tesla T4 GPU and Internet. It first runs a real page-34 smoke, then runs all 1,034 pages and creates a downloadable ZIP.

In [ ]:
# Current stable packages. PaddleOCR 3.7.0 and PaddlePaddle GPU 3.3.1.
%pip install -q 'paddleocr==3.7.0'
%pip install -q 'paddlepaddle-gpu==3.3.1' -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q 'pymupdf>=1.25,<1.26' 'pillow>=10,<12'

import os
os.environ['PADDLE_PDX_CACHE_HOME'] = '/kaggle/working/paddlex-cache'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

import paddle
import paddleocr
print({'paddle': paddle.__version__, 'paddleocr': paddleocr.__version__,
       'compiled_with_cuda': paddle.is_compiled_with_cuda(),
       'device': paddle.get_device()})
assert paddle.is_compiled_with_cuda(), 'Select a Kaggle GPU accelerator first.'

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/doc-agent-G07')
if not (REPO / 'extras/ocr_research/kaggle-paddleocr-full-book.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                    'https://github.com/smammahdi/doc-agent-G07.git', str(REPO)],
                   check=True)
SCRIPT = REPO / 'extras/ocr_research/kaggle-paddleocr-full-book.py'
assert SCRIPT.is_file(), SCRIPT
print('script:', SCRIPT)

In [ ]:
# Smoke test on one real page before spending time on the full book.
subprocess.run([sys.executable, str(SCRIPT), '--pages', '34'], check=True)

In [ ]:
# Full run: rerun this cell after an interruption; completed pages are reused.
subprocess.run([
    sys.executable, str(SCRIPT), '--pages', 'all',
    '--zip', '/kaggle/working/paddleocr-full-book-doclayout-yolo.zip',
], check=True)

In [ ]:
import json
from pathlib import Path

result = Path('/kaggle/working/paddleocr-doclayout-yolo/paddleocr')
summary = json.loads((result / 'summary.json').read_text())
print(summary)
print('download:', '/kaggle/working/paddleocr-full-book-doclayout-yolo.zip')
assert summary['status'] == 'complete' and summary['regions_error'] == 0
assert summary['pages_completed'] == 1034